# 03 · DistilBERT and aspect-based sentiment

TA Units 2–4. Fine-tune a pretrained transformer for binary sentiment, then optionally train an aspect-conditioned model on human annotations. The architecture supports both lightweight CPU inference and trained transformer inference.

In [1]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from CineScope/notebooks or CineScope.'
sys.path.insert(0, str(ROOT / 'backend'))
sys.path.insert(0, str(ROOT / 'scripts'))
os.chdir(ROOT)
(ROOT / 'artifacts').mkdir(exist_ok=True)
SEED = 42

Install `python -m pip install -e ".[transformers]"`. A GPU is recommended for full IMDb training. Model/data downloads are separate from application startup.

**Do not invent aspect labels from movie ratings.** A positive overall review can criticize pacing or music. The clause baseline is available now; the trained aspect model requires a labeled set for story, acting, visuals, music, pacing and ending.

In [2]:
RUN_SENTIMENT_TRAINING = False
if RUN_SENTIMENT_TRAINING:
    from cinescope.training import train_transformer
    trainer, metrics = train_transformer(ROOT/'data/processed/imdb_reviews.csv',
        model_dir=str(ROOT/'models/sentiment_transformer'),output=str(ROOT/'artifacts/transformer'),epochs=2,batch_size=8)
    display(metrics)
else:
    print('DistilBERT training is disabled until real data and model downloads are available.')

DistilBERT training is disabled until real data and model downloads are available.


The trainer uses the first 384 tokens for each training/evaluation example. The runtime evaluates all chunks of a long review and combines their probabilities by length. These are different inference policies: report both policies in the long-review evaluation rather than transferring the first-chunk benchmark score to the runtime.

For aspect scoring, provide one row per aspect opinion with `review_id,movie_id,aspect,text,label`. Labels are **0 negative, 1 neutral, 2 positive**. Keep absent aspects absent. Have two annotators label an overlapping subset, adjudicate disagreements, and measure agreement.

In [3]:
import pandas as pd
annotation_template = pd.DataFrame(columns=['review_id','movie_id','aspect','text','label','annotator'])
annotation_template.to_csv(ROOT/'artifacts/aspect_annotation_template.csv',index=False)
print('Template created. Collect human annotations; synthetic labels are not used for training.')

Template created. Collect human annotations; synthetic labels are not used for training.


In [4]:
RUN_ASPECT_TRAINING = False
if RUN_ASPECT_TRAINING:
    from cinescope.training import train_absa
    trainer, metrics = train_absa(ROOT/'data/annotations/aspect_labels.csv',
        model_dir=str(ROOT/'models/aspect_transformer'),output=str(ROOT/'artifacts/absa'))
    display(metrics)
else:
    print('Using clause-level aspect evidence until a validated aspect model is trained.')

Using clause-level aspect evidence until a validated aspect model is trained.


In [5]:
from cinescope.reviews import ReviewAnalyzer
model = ReviewAnalyzer()
r = model.analyze('The acting is excellent. The visuals are stunning, but the pacing is terrible. The music is not good.')
import pandas as pd
pd.DataFrame([{ 'aspect':a, **v} for a,v in r['aspects'].items()])

    aspect  ...                                      method
0    Story  ...  clause-mention + tfidf-logistic-regression
1   Acting  ...  clause-mention + tfidf-logistic-regression
2  Visuals  ...  clause-mention + tfidf-logistic-regression
3    Music  ...  clause-mention + tfidf-logistic-regression
4   Pacing  ...  clause-mention + tfidf-logistic-regression
5   Ending  ...  clause-mention + tfidf-logistic-regression

[6 rows x 6 columns]


## Meaning of the numbers
For binary models, the score is `100 × P(positive)`. For the trained three-class aspect model, it is `100 × [P(positive) + 0.5 × P(neutral)]`. The fallback uses a labeled heuristic sentiment index. Missing or uncertain evidence returns `None`, not a fabricated 90.

A movie score averages review-level aspect indices over eligible reviews, so a verbose reviewer does not dominate merely by repeating an aspect. Show the supporting review count and evidence. Genre/theme scores are separately defined as mention coverage; these are relevance cues, not film-quality ratings.